# Inference demonstration
This notebook demonstrates how to load a pre-trained multi-label classification model, embed new abstract text using the [Qwen/Qwen3-Embedding-8B](https://huggingface.co/Qwen/Qwen3-Embedding-8B), and generate domain predictions.
 
It provides two modes for abstract text entry:
1. Manual abstract text entry
2. Automated fetching via the [Heal Data Platform](https://healdata.org/portal) API endpoint using study ID.

## Libraries

In [ ]:
# Uncomment the line to install libraries used in this notebook
#!pip install -q requests joblib torch numpy transformers

# Version matching for scikit-learn
#!pip install -q scikit-learn==1.7.2


In [ ]:
import os
import requests
import joblib
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

## Inputs/outputs directories

In [ ]:
# Directory setup for inputs and outputs, change as needed
input_dir = "./inputs"
output_dir = "./outputs/"

# Create directories if they do not exist
os.makedirs(input_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

## Configuration

In [ ]:
embedding_model = "Qwen/Qwen3-Embedding-8B"

In [ ]:
domain_labels = [
    'Anxiety', 
    'Depression', 
    'Global Satisfaction with Treatment', 
    'Pain', 
    'Pain Catastrophizing', 
    'Pain Intensity', 
    'Pain Interference', 
    'Physical Functioning', 
    'Quality of Life', 
    'Sleep', 
    'Substance Use'
]

In [ ]:
# Model of interest in the input_dir, change as needed.

#model_name = "heal-basic_full_no_strat_logreg_seed2024.joblib"
#model_name = "heal-basic_collapsed_no_strat_logreg_seed2024.joblib"
#model_name = "heal-extended_full_cardinality_strat_mlp_seed21.joblib"
#model_name = "heal-extended_collapsed_minority_strat_mlp_seed30380.joblib"
#model_name = "nih-reporter_full_no_strat_mlp_seed123.joblib"
model_name = "nih-reporter_collapsed_no_strat_mlp_seed2024.joblib"

model_path = os.path.join(input_dir, model_name)


## Helper Functions

In [ ]:
def get_embedding(text, tokenizer, model, device):
    """Generates a text embedding vector using the target transformer model."""
    inputs = tokenizer(
        text, truncation=True, padding=True, return_tensors="pt", max_length=512
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    # Mean pooling across the token dimension
    embeddings = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().float().numpy()
    return embeddings

In [ ]:
def fetch_abstracts_from_heal(hdp_ids):
    """Fetches dictionary mapping HDP IDs to their abstract texts from HEAL MDS."""
    abstract_dict = {}
    for hdp_id in hdp_ids:
        url = f"https://healdata.org/mds/metadata/{hdp_id}"
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                content = r.json()
                abstract_text = content.get("nih_reporter", {}).get("abstract_text", "")
                if abstract_text:
                    abstract_dict[hdp_id] = abstract_text
                else:
                    print(f"[WARN] No abstract text found for {hdp_id}")
            else:
                print(f"Issues with downloading data for {hdp_id}: {r.status_code}")
        except Exception as e:
            print(f"Issues with downloading data for {hdp_id}: {e}")
    return abstract_dict

In [ ]:
def get_trained_labels(clf_model):
    """
    Dynamically reconstructs the correct label names by matching the model's 
    internal output structure with the training notebook's alphabetical master set.
    Supports MLP, Random Forest, and MultiOutputClassifier (Logistic Regression).
    """
    master_alphabetical_domains = [
        'Anxiety',                             # 0
        'Depression',                          # 1
        'Global Satisfaction with Treatment',  # 2
        'Pain',                                # 3
        'Pain Catastrophizing',                # 4
        'Pain Intensity',                      # 5
        'Pain Interference',                   # 6
        'Physical Functioning',                # 7
        'Quality of Life',                     # 8
        'Sleep',                               # 9
        'Substance Use'                        # 10
    ]
    
    # Handle MultiOutputClassifier (Logistic Regression)
    # This architecture uses distinct sub-estimators and drops all-zero columns
    if clf_model.__class__.__name__ == "MultiOutputClassifier":
        num_active_classes = len(clf_model.classes_)
        if num_active_classes == 10:
            return [d for d in master_alphabetical_domains if d != 'Pain']
        elif num_active_classes == 7:
            dropped_cols = {'Global Satisfaction with Treatment', 'Pain Catastrophizing', 'Pain Intensity', 'Pain Interference'}
            return [d for d in master_alphabetical_domains if d not in dropped_cols]

    # Handle Natively Multi-Label Models (MLPClassifier, RandomForestClassifier)
    # These architectures preserve all 11 output slots from the master column pool
    else:
        # Check if the internal attributes show 11 expected outputs
        if getattr(clf_model, "n_outputs_", None) == 11:
            return master_alphabetical_domains
        elif isinstance(getattr(clf_model, "classes_", None), list) and len(clf_model.classes_) == 11:
            return master_alphabetical_domains
            
    # Fallback message if configuration is unknown
    raise ValueError(
        f"Could not automatically map label dimensions for model type: {clf_model.__class__.__name__}. "
        f"Please verify this model file's target training matrix shape."
    )


In [ ]:
def predict_domains(abstracts_dict, clf_model, tokenizer, emb_model, device):
    """Runs embeddings and predicts target domains, safely adapting to any model framework."""
    clf_model_name = clf_model.__class__.__name__
    
    # Resolve the exact text mapping list matching this model file
    active_labels = get_trained_labels(clf_model)
    
    print(f"\n--- Generating Predictions using {clf_model_name} ({len(active_labels)} Labels) ---\n")
    for item_id, text in abstracts_dict.items():
        # Generate text embedding
        emb = get_embedding(text, tokenizer, emb_model, device)
        X_input = np.array([emb])
        
        # Run classification prediction logic
        preds = clf_model.predict(X_input)[0]  # Grab the first row output mask
        
        # Map active index positions safely to the dynamically resolved labels list
        predicted_domains = [active_labels[i] for i, val in enumerate(preds) if val == 1]
        
        print(f"ID: {item_id}")
        print(f"Abstract Snippet: {text[:85]}...")
        print(f"Predicted Domains: {predicted_domains if predicted_domains else 'None'}")
        print("-" * 50)

## Model and Tokenizer Initialization

In [ ]:
print(f"Loading classification model from: {model_path}")
clf_model = joblib.load(model_path)


In [ ]:
print(f"Loading embedding model: {embedding_model}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(embedding_model, trust_remote_code=True)
model = AutoModel.from_pretrained(embedding_model, trust_remote_code=True).to(device)
model.eval()

## Inference examples

### Example 1: Process manually submitted abstracts

In [ ]:
# Define abstracts for predictions, change as needed
manual_abstracts = {
    "study_01": "PROJECT SUMMARY. This study focuses on treating chronic musculoskeletal pain, "
                "and reducing opioid reliance in adult patients.",
    "study_02": "PROJECT ABSTRACT. This project tracks how stabilizing REM sleep architecture "
                "using a targeted sleep protocol accelerates the reduction of anhedonia in patients "
                "undergoing treatment for major depression."
}


In [ ]:
# Run predictions
predict_domains(manual_abstracts, clf_model, tokenizer, model, device)

### Example 2: Process abstracts fetched from [Heal Data Platform](https://healdata.org/portal) API endpoint

In [ ]:
# Define study ids for predictions, change as needed
hdp_ids = [
    "HDP00049",
    "HDP00007"
]

In [ ]:
# Get abstracts
print(f"Fetching {len(hdp_ids)} abstracts from HEAL API...")
api_abstracts = fetch_abstracts_from_heal(hdp_ids)

In [ ]:
# Run predictions
predict_domains(api_abstracts, clf_model, tokenizer, model, device)